In [2]:
import pandas as pd
import random

macbeth_df = pd.read_fwf('macbeth.txt')
macbeth_df.columns = ['text']

In [3]:
macbeth_df

,text
0,THE TRAGEDY OF MACBETH
1,by William Shakespeare
2,Dramatis Personae
3,"DUNCAN, King of Scotland"
4,"MACBETH, Thane of Glamis and Cawdor, a general..."
...,...
2660,"We will perform in measure, time, and place."
2661,"So thanks to all at once and to each one,"
2662,Whom we invite to see us crown'd at Scone.
2663,Flourish. Exeunt.


In [4]:
import numpy as np

splits = np.array_split(macbeth_df, 5)

# Split 0 -> Caesar Cipher

In [5]:
def caesar_encrypt(text, s=7):
    result = ""
    # transverse the plain text
    for i in range(len(text)):
        char = text[i]
        # Encrypt uppercase characters in plain text
        
        if (char.isupper()):
            result += chr((ord(char) + s-65) % 26 + 65)
        # Encrypt lowercase characters in plain text
        else:
            result += chr((ord(char) + s - 97) % 26 + 97)
    return result

In [6]:
caesar_df = pd.DataFrame(columns=['algorithm', 'encrypt'])
caesar_df['encrypt'] = [caesar_encrypt(text, random.randint(0, 26)) for text in splits[0].text.values] 
caesar_df['algorithm'] = 'caesar'

In [7]:
caesar_df

,algorithm,encrypt
0,caesar,BPMvBZIOMLGvWNvUIKJMBP
1,caesar,dapYknnkcopUjcmgurgctg
2,caesar,IwfrfynxsUjwxtsfj
3,caesar,BSLAYLxlIglelmdlQamrjylb
4,caesar,VJLKNCQiwCqjwnwxowPujvrbwjwmwLjfmxaiwjwpnwnaju...
...,...,...
528,caesar,Nzkyeyzjejlitvrjvqejlttvjjfekyrkeslkekyzjescfn
529,caesar,Yustfznqzftqznqmmxxzmzpzftqzqzpmmxxzmtqdql
530,caesar,Dwvpjgtgbpwrqppvjkupdcpmpcpfpujqcnpqhpvkogb
531,caesar,MukbtdzkcfdjxudbyvudjedsecurdRkjdyddjxuiudsqiui


# Split 1 -> Affine Cipher

## Code for Affine Cipher

In [8]:
class Affine(object):
    def __init__(self):
        pass

    def egcd(self, a, b): 
        x,y, u,v = 0,1, 1,0
        while a != 0: 
            q, r = b//a, b%a 
            m, n = x-u*q, y-v*q 
            b,a, x,y, u,v = a,r, u,v, m,n 
        gcd = b 
        return gcd, x, y 

    def modinv(self, a, m): 
        gcd, x, y = self.egcd(a, m) 
        if gcd != 1: 
            return None # modular inverse does not exist 
        else: 
            return x % m 
    
    def encrypt(self, text, key): 
        #E = (a*x + b) % 26 
        return ''.join([ chr(((key[0]*(ord(t) - ord('A')) + key[1] ) % 26) + ord('A')) for t in text.upper().replace(' ', '') ]) 


    def decrypt(self, cipher, key): 
        #D(E) = (a^-1 * (E - b)) % 26
        return ''.join([ chr(((self.modinv(key[0], 26)*(ord(c) - ord('A') - key[1])) % 26) + ord('A')) for c in cipher ]) 
      
affine = Affine()

In [9]:
affine_df = pd.DataFrame(columns=['algorithm', 'encrypt'])
affine_df['encrypt'] = [affine.encrypt(text, (random.randint(0, 26), random.randint(0, 26))) for text in splits[1].text.values]
affine_df['algorithm'] = 'affine'

In [10]:
affine_df

,algorithm,encrypt
0,affine,AMAASMYEKGOCWGYAEKKUCYWCAOYEGGECGCGOOGCOE
1,affine,SOUWISYGSYGEIEGISOGYSYEMGEGISYIIAGAKYMSEUG
2,affine,PBZZRNDFTHRVNTRRDVRNTFBFBHRPBVFBNNDPHNLVPR
3,affine,YPPFKPTIUZWRSSXVRSXKXZIQPFCUXYKFRYY
4,affine,UUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUU
...,...,...
528,affine,RZZDNJJBLDZNJBBDJJVNBRJZZJNHVPDHZPNLHZVNZLDZV
529,affine,XDPPLYXNJIIVGKYDGNQNMLEXDPPLYXNJIENFMYNQTMNDWN...
530,affine,AKKWWEKQGYQEGYKQKAAOKUOMYSOYUQWIKOKAOKIKIOKKAG...
531,affine,RDONNNUVDPEPNU


# Split 2 -> Vigenere Cipher

In [59]:
MIN_KEY_LEN = 3
MAX_KEY_LEN = 10
ALPHA = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

class Vigenere:
    def __init__(self, key=None):
        if key is None:
            key = self.generate_random_key()
        self.key = key

    def get_key(self):
        return self.key
    
    def set_key(self):
        key = self.generate_random_key()

    def generate_random_key(self) -> str:
        length = random.randint(MIN_KEY_LEN, MAX_KEY_LEN + 1)
        return ''.join(random.sample(ALPHA, length))

    def generate_keyword(self, n: int) -> str:
        return ((n // len(self.key) + 1) * self.key)[:n]

    def get_ord(self, letter):
        return ALPHA.index(letter.upper())
    
    def get_chr(self, num, uppercase=True):
        return ALPHA[num % 26] if uppercase else ALPHA[num % 26].lower()
    
    def encrypt(self, plaintext: str) -> str:
        keyword = self.generate_keyword(len(plaintext))
        return ''.join([self.get_chr(self.get_ord(ch) + self.get_ord(keyword[i]), ch.isupper())
                        if ch.upper() in ALPHA else ch for i, ch in enumerate(plaintext)])

    def decrypt(self, ciphertext: str) -> str:
        keyword = self.generate_keyword(len(ciphertext))
        return ''.join([self.get_chr(self.get_ord(ch) - self.get_ord(keyword[i]), ch.isupper())
                        if ch.upper() in ALPHA else ch for i, ch in enumerate(ciphertext)])

In [60]:
vigenere_test = Vigenere('TESTING')
plaintext = 'this is a test. Hello!'
ciphertext = vigenere_test.encrypt(plaintext)
print(plaintext)
print(ciphertext)

vigenere_test.decrypt(ciphertext)

this is a test. Hello!
mlal vy e mmfz. Zxtyu!


'this is a test. Hello!'

In [61]:
vigenere = Vigenere()

vigenere_df = pd.DataFrame(columns=['algorithm', 'encrypt'])
for text in splits[2].text.values:
    vigenere.set_key()
    vigenere_df['encrypt'] = [vigenere.encrypt(text) for text in splits[2].text.values]
vigenere_df['algorithm'] = 'vigenere'

vigenere_df

,algorithm,encrypt
0,vigenere,Eonkr Vgnrou.
1,vigenere,"BBHWUP. Tiia iuyt cz oic: Eonh, Cbqjos, Gmusit..."
2,vigenere,"At zhf cejlj xiseo vrpgosfx, bhj J lebl"
3,vigenere,Tiia qfgy'xyt gusu lovfry zur'n; zyz jn wbm sbcj
4,vigenere,"Iu yhpord hut mzaox io zhz votnkrjne,"
...,...,...
528,vigenere,Hbnn ti eyuypflgtf zhf Qioa tiuz iy
529,vigenere,Psyvasyy gix tise uztfgvt il xux.
530,vigenere,LFHTOY. Sfhz iy tp Sadxafg?
531,vigenere,"LPLJ. Bk ecj, utd qoti gn uhspfatf ""Sjl, oiz J,"""


# Split 4 -> Railfence Cipher

In [62]:
class RailFence(object):
    def __init__(self):
        pass

    def matrix(self, text, key): 
        try:
            rail = [['\n' for i in range(len(text))] for j in range(key)]
            dir_down = False
            row, col = 0, 0

            for i in range(len(text)):
                if (row == 0) or (row == key - 1):
                    dir_down = not dir_down
                rail[row][col] = text[i]
                col += 1
                if dir_down:
                    row += 1
                else:
                    row -= 1
        except:
            print("ERROR", text, key)

        return rail
    
    def encrypt(self, text, key): 

        rail=self.matrix(text,key)
        result = []
        for i in range(key):
            for j in range(len(text)):
                if rail[i][j] != '\n':
                    result.append(rail[i][j])
        return("" . join(result))

        
rail_fence = RailFence()

In [63]:
rail_fence_df = pd.DataFrame(columns=['algorithm', 'encrypt'])
rail_fence_df['encrypt'] = [rail_fence.encrypt(text, random.randint(2, len(text))) for text in splits[4].text.values]
rail_fence_df['algorithm'] = 'rail_fence'


In [64]:
rail_fence_df

,algorithm,encrypt
0,rail_fence,That Christendom gives .otu
1,rail_fence,RIeO wrSdcsSlon.uua ol Wd
2,rail_fence,This comfort witshd rtohwe elviakhe !I Btu
3,rail_fence,"That would be howl'd out in, rtihae tdrees"
4,rail_fence,Wi hrndnhteaglochre utteehso am hl.
...,...,...
528,rail_fence,"We will perform .ienc amlepa sdunrae ,, etmi"
529,rail_fence,"So thanks t,oe naol lh caate oontc ed na"
530,rail_fence,Wio eh nt scdan.oev sur'tomwiee on c tewS
531,rail_fence,Flourish. Exeun.t


# Exporting to CSV

In [65]:
def export_df(df, filename):
    df.to_csv(filename, index=False)

# combine all the dataframes
combined_df = pd.concat([caesar_df, affine_df, vigenere_df, rail_fence_df])

# export the combined dataframe
export_df(combined_df, 'dataset.csv')